# Phase 3: LoRA fine-tuning of Qwen2-VL-2B on native-Windows GUI grounding (v2)

Second real training run for the `computer-use` project's GUI grounding model (ADR-0003). Runs on Kaggle's free T4/P100 GPU quota, matching the project's $0-budget commitment.

**Honest scope, stated up front**: this trains on the **v2 dataset** -- 570 examples across all 14/14 registry apps, collected 2026-07-16 after fixing four real bugs found in the v1 pipeline (chrome-vs-content sampling bias, a UIA tree-walk depth cutoff too shallow for packaged/WinUI3 apps, a packaged-app rect-not-ready race, and two PII leaks -- see `data/gui_grounding/README.md`). The held-out pool now covers all 4 held-out apps (Character Map, Device Manager, Audacity, Notepad++), so H3's generalization claim finally has real diversity behind it -- but this run is still an engineering validation (does the fixed pipeline train end-to-end, is the loss sane), not the final ablation study.

Every piece below (`prepare_dataset.py`, `chat_format.py`, `lora_config.py`, `dataset.py`, `train_lora.py`) was built and verified locally (103 passing tests, plus live checks against the real tokenizer/processor/model architecture on the `meta` device) before this notebook was written -- see `docs/journal.md` for that verification trail. The v1 run already proved this notebook touches real model weights/GPU correctly; this run's job is to confirm the same holds on the corrected, full-coverage dataset.

## 1. Confirm GPU is attached

In Kaggle: Settings (right panel) -> Accelerator -> GPU T4 x2 (or P100). Must be set before running anything below.

In [ ]:
!nvidia-smi

## 2. Get the code

Clones the public `computer-use` repo and installs it with the `training` extra (`transformers`, `torch`, `torchvision`, `peft`, `jinja2` -- see `pyproject.toml`).

In [ ]:
import sys

!git clone https://github.com/rudranaresh0201/computer-use.git
%cd computer-use
# Use {sys.executable} -m pip, not bare `pip` -- Kaggle images can have more
# than one Python/pip on PATH, and a bare `!pip install` has been observed to
# install into a different environment than the one this kernel is actually
# running, producing "ModuleNotFoundError: No module named 'computeruse'" on
# the very next cell even though the install itself reported success.
# Also dropped `-q` here deliberately: a silent install failure (dependency
# conflict, etc.) is exactly what would cause the same downstream error, and
# `-q` was hiding whichever of the two causes was actually happening.
!{sys.executable} -m pip install -e ".[training]"

In [ ]:
import importlib

importlib.invalidate_caches()
import computeruse
print("computeruse imported OK from", computeruse.__file__)

In [ ]:
import sys

# Kaggle's base image preinstalls torchao (0.10.0 as of 2026-07-16), and
# peft's LoRA dispatcher unconditionally checks its version -- if present
# but below peft's required minimum, it raises ImportError even though we
# never use torchao anywhere in this project. If it's simply not installed,
# peft's is_torchao_available() returns False and quietly falls through to
# the LoRA dispatch path we actually want, so uninstalling is the fix, not
# upgrading (confirmed working 2026-07-16).
!{sys.executable} -m pip uninstall -y -q torchao

from pathlib import Path

from computeruse.training.dataset import resolve_path

# Kaggle's actual mount path nests under datasets/<owner>/<slug>, not flat
# /kaggle/input/<slug> as the "+ Add Input" panel implies -- confirmed
# 2026-07-16 (os.walk("/kaggle/input") showed /kaggle/input/datasets/
# <username>/<slug>/labels.jsonl). This searches for labels.jsonl instead of
# hardcoding one convention, so it works regardless of which path shape your
# Kaggle environment actually uses.
_candidates = list(Path("/kaggle/input").rglob("labels.jsonl"))
assert _candidates, (
    "no labels.jsonl found anywhere under /kaggle/input -- check the v2 "
    "dataset is actually attached via '+ Add Input' in the right panel"
)
assert len(_candidates) == 1, (
    f"found multiple labels.jsonl under /kaggle/input: {_candidates} -- "
    "detach any old/extra dataset versions so there's no ambiguity"
)
KAGGLE_DATASET_ROOT = _candidates[0].parent
OUTPUT_DIR = Path("/kaggle/working/lora_grounder")

# resolve_path checks both the nested path and a flat fallback -- see note
# in the previous markdown cell about Kaggle's web uploader flattening
train_split_path = resolve_path(KAGGLE_DATASET_ROOT, "splits/train.jsonl")
print("dataset root looks correct:", KAGGLE_DATASET_ROOT)
print("train split found at:", train_split_path)

In [ ]:
from pathlib import Path

from computeruse.training.dataset import resolve_path

KAGGLE_DATASET_ROOT = Path("/kaggle/input/gui-grounding-v2")  # <-- adjust to your uploaded dataset's slug
OUTPUT_DIR = Path("/kaggle/working/lora_grounder")

assert (KAGGLE_DATASET_ROOT / "labels.jsonl").exists(), (
    f"labels.jsonl not found under {KAGGLE_DATASET_ROOT} -- check the dataset is attached "
    "and KAGGLE_DATASET_ROOT matches its slug"
)
# resolve_path checks both the nested path and a flat fallback -- see note above
train_split_path = resolve_path(KAGGLE_DATASET_ROOT, "splits/train.jsonl")
print("dataset root looks correct:", KAGGLE_DATASET_ROOT)
print("train split found at:", train_split_path)

## 4. Build the trainer

Loads the real Qwen2-VL-2B-Instruct weights (~4GB download, first cell run only), attaches the verified LoRA config (0.829% trainable params against the real architecture -- see `training/lora_config.py`), and wires up `GroundingDataset`/`collate_fn` against the v2 train/dev splits.

In [ ]:
from computeruse.training.train_lora import build_trainer

trainer = build_trainer(KAGGLE_DATASET_ROOT, OUTPUT_DIR)

## 5. Train

1 epoch over 252 train examples (v2's actual size) is fast even on a single T4 -- this is deliberately a small first run, not the final tuned config. Watch the eval (dev) loss printed each epoch; per the hypothesis doc, **only the dev split is allowed to drive any tuning** -- do not look at `test_held_out_app` or `test_same_app` numbers to make decisions here.

In [ ]:
train_result = trainer.train()
print(train_result)

In [ ]:
trainer.save_model(str(OUTPUT_DIR / "final"))
print("saved LoRA adapter to", OUTPUT_DIR / "final")

## 6. Sanity-check a few predictions

Not a real evaluation (that's the 4-arm ablation, not built yet) -- just a first look at whether the model outputs something plausible at all: a well-formed `(x,y)` string in the right numeric range, before any accuracy metric is computed.

In [ ]:
import json

from PIL import Image
from transformers import AutoProcessor

from computeruse.training.chat_format import to_conversation
from computeruse.training.dataset import resolve_path
from computeruse.training.prepare_dataset import TrainingExample

processor = AutoProcessor.from_pretrained("Qwen/Qwen2-VL-2B-Instruct")
model = trainer.model
model.eval()

dev_split_path = resolve_path(KAGGLE_DATASET_ROOT, "splits/dev.jsonl")
dev_examples = [
    TrainingExample(**json.loads(line))
    for line in dev_split_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
][:5]

print(f"spot-checking {len(dev_examples)} dev examples (not a metric, just a look)")
for ex in dev_examples:
    image_path = resolve_path(KAGGLE_DATASET_ROOT, ex.image_path)
    image = Image.open(image_path).convert("RGB")
    prompt_only = to_conversation(ex)[:-1]  # drop the ground-truth assistant turn
    prompt_text = processor.tokenizer.apply_chat_template(
        prompt_only, tokenize=False, add_generation_prompt=True
    )
    inputs = processor(text=[prompt_text], images=[image], return_tensors="pt").to(model.device)
    generated = model.generate(**inputs, max_new_tokens=16)
    prediction = processor.tokenizer.decode(
        generated[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    )
    print(f"{ex.app:12s} {ex.prompt!r:35s} predicted={prediction!r}  ground_truth={ex.target!r}")

## Next steps (not this notebook)

1. ~~Collect the 6 blocked apps~~ -- done 2026-07-16: all 14/14 registry apps now collected, v2 dataset frozen.
2. Build the 4 evaluation arms (UIA-only, zero-shot VLM, fine-tuned grounder, hybrid) -- none of that exists yet.
3. Only then run the one final held-out evaluation and report H1-H3, sliced 3 ways, honestly.